In [ ]:
%env WORKDIR=/tmp/vault            

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

## Install PostgreSQL

In [ ]:
%%bash

export DOCKER_CONFIG=$HOME/.config/containers

# Add Helm repository by Bitnami
helm repo add bitnami https://charts.bitnami.com/bitnami

In [ ]:
%%bash
set -euo pipefail

kubectl apply -f manifest/local-pv.yaml
kubectl apply -f manifest/pv-claim.yaml

helm upgrade --install postgresql-dev \
  -f manifest/postgres.yaml \
  oci://registry-1.docker.io/bitnamicharts/postgresql \
  -n vault \
  --set volumePermissions.enabled=true \
  --wait \
  --timeout 10m


In [ ]:
%%bash

if ! kubectl wait --namespace vault \
  --for=condition=Ready pod/postgresql-dev-0 \
  --timeout=5m; then
  kubectl describe pod -n vault postgresql-dev-0
  exit 1
fi

kubectl get pod -n vault postgresql-dev-0 -o wide

Create role that will be used by Vault

In [ ]:
! kubectl exec -it postgresql-dev-0 -n vault -- sh -c 'export PGPASSWORD=StrongPassword; psql --host 127.0.0.1 -U postgres -c "CREATE ROLE \"ro\" NOINHERIT;"'
! kubectl exec -it postgresql-dev-0 -n vault -- sh -c 'export PGPASSWORD=StrongPassword; psql --host 127.0.0.1 -U postgres -c "GRANT SELECT ON ALL TABLES IN SCHEMA public TO \"ro\";"'

## Enable the database secret engine

In [ ]:
%%bash
export POSTGRES_URL=postgresql-dev.vault.svc.cluster.local
# Enable engine
vault secrets enable database

# Configure the database secrets engine with the connection credentials for the Postgres database.
vault write database/config/postgresql \
     plugin_name=postgresql-database-plugin \
     connection_url="postgresql://{{username}}:{{password}}@$POSTGRES_URL/postgres?sslmode=disable" \
     allowed_roles=* \
     username="postgres" \
     password="StrongPassword"

## Create a role in Vault

In [ ]:
%%bash
cat > ${WORKDIR}/readonly.sql <<EOF
CREATE ROLE "{{name}}" WITH LOGIN PASSWORD '{{password}}' VALID UNTIL '{{expiration}}' INHERIT;
GRANT ro TO "{{name}}";
EOF

In [ ]:
%%bash
vault write database/roles/readonly \
      db_name=postgresql \
      creation_statements=@${WORKDIR}/readonly.sql \
      default_ttl=60s \
      max_ttl=200s

## Request Credentials

In [ ]:
! vault read -format=json database/creds/readonly 

In [ ]:
! vault lease revoke -prefix database/creds/readonly

## Rotate Root password

In [ ]:
! vault write -force database/rotate-root/postgresql

# JBOSS/WildFly configuration with Vault Agent and Dynamic Database Credentials

In [ ]:
%%bash
set -euo pipefail

export JAVA_HOME=/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home
export PATH="${JAVA_HOME}/bin:${PATH}"

DEMO_WORKDIR=/tmp/vault-jboss-db-demo
AGENT_DIR="${DEMO_WORKDIR}/agent"
SECRETS_DIR="${DEMO_WORKDIR}/secrets"
WILDFLY_DIR="${DEMO_WORKDIR}/wildfly"
LOG_DIR="${DEMO_WORKDIR}/logs"
WILDFLY_VERSION=36.0.1.Final
WILDFLY_HOME="${WILDFLY_DIR}/wildfly-${WILDFLY_VERSION}"
RELOAD_SCRIPT="${WILDFLY_DIR}/reload-wildfly.sh"

mkdir -p "${AGENT_DIR}" "${SECRETS_DIR}" "${WILDFLY_DIR}" "${LOG_DIR}"

cat > "${DEMO_WORKDIR}/jboss-db-read.hcl" <<'EOF'
path "database/creds/readonly" {
  capabilities = ["read"]
}
EOF
vault policy write jboss-db-read "${DEMO_WORKDIR}/jboss-db-read.hcl"

if ! vault auth list -format=json | jq -e 'has("approle/")' >/dev/null; then
  vault auth enable approle
fi
vault write auth/approle/role/jboss-db-demo \
  token_policies=jboss-db-read \
  token_ttl=5m \
  token_max_ttl=30m \
  secret_id_ttl=24h
vault read -field=role_id auth/approle/role/jboss-db-demo/role-id > "${AGENT_DIR}/role_id"
vault write -field=secret_id -f auth/approle/role/jboss-db-demo/secret-id > "${AGENT_DIR}/secret_id"
chmod 600 "${AGENT_DIR}/role_id" "${AGENT_DIR}/secret_id"

if [ ! -x "${WILDFLY_HOME}/bin/standalone.sh" ]; then
  curl -fsSL -o "${WILDFLY_DIR}/wildfly-${WILDFLY_VERSION}.zip" "https://github.com/wildfly/wildfly/releases/download/${WILDFLY_VERSION}/wildfly-${WILDFLY_VERSION}.zip"
  unzip -q "${WILDFLY_DIR}/wildfly-${WILDFLY_VERSION}.zip" -d "${WILDFLY_DIR}"
fi

cat > "${WILDFLY_DIR}/start-wildfly.sh" <<EOF
#!/bin/sh
set -eu
export JAVA_HOME="${JAVA_HOME}"
export PATH="\${JAVA_HOME}/bin:\${PATH}"
. "${SECRETS_DIR}/datasource.env"
export JAVA_OPTS="\${JAVA_OPTS:-} -Ddemo.db.username=\${DB_USERNAME} -Ddemo.db.password=\${DB_PASSWORD}"

DEPLOYMENT_DIR="${WILDFLY_HOME}/standalone/deployments/vault-db-demo.war"
mkdir -p "\${DEPLOYMENT_DIR}/WEB-INF"
cat > "\${DEPLOYMENT_DIR}/WEB-INF/jboss-web.xml" <<'WEBEOF'
<?xml version="1.0" encoding="UTF-8"?>
<jboss-web xmlns="http://www.jboss.com/xml/ns/javaee" version="8.0">
  <context-root>/vault-db-demo</context-root>
</jboss-web>
WEBEOF
cat > "\${DEPLOYMENT_DIR}/index.html" <<HTML
<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><title>Vault Database Credentials</title></head>
<body>
  <h1>Vault Agent + WildFly Database Secrets Engine</h1>
  <p>Dynamic database username: <code>\${DB_USERNAME}</code></p>
  <p>Dynamic database password: <code>\${DB_PASSWORD}</code></p>
</body>
</html>
HTML
touch "\${DEPLOYMENT_DIR}.dodeploy"

exec "${WILDFLY_HOME}/bin/standalone.sh" -b 127.0.0.1 \
  -Djboss.http.port=18080 \
  -Djboss.https.port=18443 \
  -Djboss.management.http.port=19990
EOF
chmod 700 "${WILDFLY_DIR}/start-wildfly.sh"

cat > "${RELOAD_SCRIPT}" <<'EOF'
#!/bin/sh
set -eu

DEMO_WORKDIR=/tmp/vault-jboss-db-demo
LOCK_DIR="${DEMO_WORKDIR}/wildfly/reload.lock"
LOG_FILE="${DEMO_WORKDIR}/logs/wildfly.log"

if ! mkdir "${LOCK_DIR}" 2>/dev/null; then
  exit 0
fi
trap 'rmdir "${LOCK_DIR}"' EXIT

pkill -f "${DEMO_WORKDIR}/wildfly/wildfly-" 2>/dev/null || true
for attempt in $(seq 1 60); do
  if ! lsof -nP -iTCP:18080 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:18443 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:19990 -sTCP:LISTEN >/dev/null 2>&1; then
    break
  fi
  sleep 1
done

: > "${LOG_FILE}"
nohup "${DEMO_WORKDIR}/wildfly/start-wildfly.sh" > "${LOG_FILE}" 2>&1 &
echo $! > "${DEMO_WORKDIR}/wildfly/wildfly.pid"
for attempt in $(seq 1 60); do
  grep -q 'WFLYSRV0025' "${LOG_FILE}" && break
  sleep 1
done
grep -q 'WFLYSRV0025' "${LOG_FILE}"
EOF
chmod 700 "${RELOAD_SCRIPT}"

cat > "${AGENT_DIR}/agent.hcl" <<EOF
exit_after_auth = false
pid_file = "${AGENT_DIR}/vault-agent.pid"

vault {
  address = "${VAULT_ADDR}"
  ca_cert = "${VAULT_CACERT}"
  tls_server_name = "${VAULT_TLS_SERVER_NAME}"
}

auto_auth {
  method "approle" {
    mount_path = "auth/approle"
    config = {
      role_id_file_path = "${AGENT_DIR}/role_id"
      secret_id_file_path = "${AGENT_DIR}/secret_id"
      remove_secret_id_file_after_reading = false
    }
  }

  sink "file" {
    config = { path = "${AGENT_DIR}/token" }
  }
}

template {
  destination = "${SECRETS_DIR}/datasource.env"
  contents = <<EOH
{{- with secret "database/creds/readonly" -}}
DB_USERNAME={{ .Data.username }}
DB_PASSWORD={{ .Data.password }}
{{- end }}
EOH

  exec {
    command = ["/bin/sh", "${RELOAD_SCRIPT}"]
    timeout = "90s"
  }
}
EOF

if [ -f "${AGENT_DIR}/vault-agent.pid" ] && kill -0 "$(cat "${AGENT_DIR}/vault-agent.pid")" 2>/dev/null; then
  kill "$(cat "${AGENT_DIR}/vault-agent.pid")"
fi
rm -rf "${WILDFLY_DIR}/reload.lock"
: > "${LOG_DIR}/vault-agent.log"
nohup vault agent -config="${AGENT_DIR}/agent.hcl" > "${LOG_DIR}/vault-agent.log" 2>&1 &
echo $! > "${AGENT_DIR}/vault-agent.pid"

for attempt in $(seq 1 90); do
  if [ -s "${SECRETS_DIR}/datasource.env" ] && grep -q 'WFLYSRV0025' "${LOG_DIR}/wildfly.log"; then
    break
  fi
  sleep 1
done

grep -q 'WFLYSRV0025' "${LOG_DIR}/wildfly.log"
curl -fsS http://127.0.0.1:18080/vault-db-demo/ | grep -E 'Dynamic database (username|password):'
echo 'WildFly usa credenciales dinamicas emitidas por Vault Database Secrets Engine.'

## Clean up

In [ ]:
%%bash
# Run only to clean up
# Uncomment %%bash to execute the cleanup
set -euo pipefail

if [ -f /tmp/vault-jboss-db-demo/agent/vault-agent.pid ] && kill -0 "$(cat /tmp/vault-jboss-db-demo/agent/vault-agent.pid)" 2>/dev/null; then
  kill "$(cat /tmp/vault-jboss-db-demo/agent/vault-agent.pid)"
fi
pkill -f '/tmp/vault-jboss-db-demo/wildfly/wildfly-' 2>/dev/null || true
vault lease revoke -prefix database/creds/readonly || true
vault policy delete jboss-db-read || true
vault delete auth/approle/role/jboss-db-demo || true
rm -rf /tmp/vault-jboss-db-demo
vault secrets disable database
helm uninstall postgresql-dev -n vault
kubectl delete pvc postgresql-data-claim -n vault
kubectl delete pv postgresql-data